In [76]:

import kagglehub
import pandas as pd
import os

In [ ]:
path = kagglehub.dataset_download("xfkzujqjvx97n/football-datasets")
tables = {}
print("Path to dataset files:", path)
for root, dirs, files in os.walk(path):
    for f in files:
        print(os.path.relpath(os.path.join(root, f), path))
        if f.endswith(".csv"):
            nom = os.path.splitext(f)[0]
            tables[nom] = pd.read_csv(os.path.join(root, f), low_memory=False)


Path to dataset files: /Users/charles/.cache/kagglehub/datasets/xfkzujqjvx97n/football-datasets/versions/2
player_profiles/player_profiles.csv


/var/folders/3c/z4rtw78n6hn3xy75vxt7kxbw0000gn/T/ipykernel_35916/1436188048.py:9: DtypeWarning: Columns (29: third_club_url, 30: third_club_name, 31: fourth_club_url, 32: fourth_club_name) have mixed types. Specify dtype option on import or set low_memory=False.
  tables[nom] = pd.read_csv(os.path.join(root, f))


player_national_performances/player_national_performances.csv
player_performances/player_performances.csv
player_injuries/player_injuries.csv
team_children/team_children.csv
team_competitions_seasons/team_competitions_seasons.csv
player_latest_market_value/player_latest_market_value.csv
player_market_value/player_market_value.csv
player_teammates_played_with/player_teammates_played_with.csv
team_details/team_details.csv
transfer_history/transfer_history.csv


In [78]:
for nom, df in tables.items():
    print(f"\n=== {nom} ===")
    print(df.shape)
    print(df.columns.tolist())


=== player_profiles ===
(92671, 34)
['player_id', 'player_slug', 'player_name', 'player_image_url', 'name_in_home_country', 'date_of_birth', 'place_of_birth', 'country_of_birth', 'height', 'citizenship', 'is_eu', 'position', 'main_position', 'foot', 'current_club_id', 'current_club_name', 'joined', 'contract_expires', 'outfitter', 'social_media_url', 'player_agent_id', 'player_agent_name', 'contract_option', 'date_of_last_contract_extension', 'on_loan_from_club_id', 'on_loan_from_club_name', 'contract_there_expires', 'second_club_url', 'second_club_name', 'third_club_url', 'third_club_name', 'fourth_club_url', 'fourth_club_name', 'date_of_death']

=== player_national_performances ===
(92701, 9)
['player_id', 'team_id', 'matches', 'goals', 'shirt_number', 'debut', 'coach_id', 'debut_game_id', 'career_state']

=== player_performances ===
(1878719, 20)
['player_id', 'season_name', 'competition_id', 'competition_name', 'team_id', 'team_name', 'nb_in_group', 'nb_on_pitch', 'goals', 'assist

In [105]:
# Test de l'unicité des clés primaires
for t, k in [("player_profiles", "player_id"), ("team_details", "club_id"),
             ("player_latest_market_value", "player_id")]:
    print(f"{t}.{k} unique :", tables[t][k].is_unique)

player_profiles.player_id unique : True
team_details.club_id unique : True
player_latest_market_value.player_id unique : True


In [ ]:
def check_lien(src, col, ref, ref_col):
    s, r = tables[src][col].dropna(), tables[ref][ref_col].dropna()
    return {"lien": f"{src}.{col} → {ref}.{ref_col}",
            "% distincts": round(pd.Series(s.unique()).isin(r).mean() * 100, 1),
            "% lignes": round(s.isin(r).mean() * 100, 1)}

liens = [
    ("player_performances", "player_id", "player_profiles", "player_id"),
    ("player_performances", "team_id", "team_details", "club_id"),
    ("transfer_history", "player_id", "player_profiles", "player_id"),
    ("transfer_history", "from_team_id", "team_details", "club_id"),
    ("transfer_history", "to_team_id", "team_details", "club_id"),
    ("player_market_value", "player_id", "player_profiles", "player_id"),
    ("player_latest_market_value", "player_id", "player_profiles", "player_id"),
    ("player_profiles", "current_club_id", "team_details", "club_id"),
    ("player_teammates_played_with", "teammate_player_id", "player_profiles", "player_id"),
    ("team_children", "parent_team_id", "team_details", "club_id"),
    ("team_children", "child_team_id", "team_details", "club_id"),
    ("player_national_performances", "player_id", "player_profiles", "player_id"),
]
display(pd.DataFrame([check_lien(*l) for l in liens]))

,lien,% distincts,% lignes
0,player_performances.player_id → player_profile...,100.0,100.0
1,player_performances.team_id → team_details.clu...,16.0,76.3
2,transfer_history.player_id → player_profiles.p...,77.3,76.2
3,transfer_history.from_team_id → team_details.c...,5.1,48.0
4,transfer_history.to_team_id → team_details.clu...,5.7,50.1
5,player_market_value.player_id → player_profile...,100.0,100.0
6,player_latest_market_value.player_id → player_...,100.0,100.0
7,player_profiles.current_club_id → team_details...,15.6,34.9
8,player_teammates_played_with.teammate_player_i...,34.7,67.8
9,team_children.parent_team_id → team_details.cl...,100.0,100.0


### `player_profiles`

In [80]:
print("Proportion de valeurs manquantes par variables dans la table Player_profiles")
pd.DataFrame({
    "nb" : tables["player_profiles"].isna().sum(),
    "pct" : (tables["player_profiles"].isna().mean() * 100 ).round(2)
}).sort_values("pct")


Proportion de valeurs manquantes par variables dans la table Player_profiles


,nb,pct
player_id,0,0.00
player_slug,0,0.00
player_image_url,0,0.00
current_club_name,0,0.00
current_club_id,0,0.00
main_position,2,0.00
height,4,0.00
is_eu,0,0.00
position,2,0.00
citizenship,57,0.06


In [81]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_profiles"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_profiles"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_profiles"].columns[tables["player_profiles"].isna().all()])


Nombre de lignes avec au moins une valeurs manquante :  92671
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


### `team_details`

In [82]:
print(tables["team_details"].shape)
print("Proportion de valeurs manquantes par variables dans la table team_details")
pd.DataFrame({
    "nb" : tables["team_details"].isna().sum(),
    "pct" : (tables["team_details"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(2175, 12)
Proportion de valeurs manquantes par variables dans la table team_details


,nb,pct
club_id,0,0.00
club_slug,0,0.00
club_name,0,0.00
logo_url,0,0.00
country_name,0,0.00
season_id,0,0.00
source_url,0,0.00
_last_modified_at,0,0.00
competition_id,506,23.26
competition_slug,506,23.26


In [83]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["team_details"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["team_details"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["team_details"].columns[tables["team_details"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  506
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


### `team_children`

In [85]:
print(tables["team_children"].shape)
print("Proportion de valeurs manquantes par variables dans la table team_children")
pd.DataFrame({
    "nb" : tables["team_children"].isna().sum(),
    "pct" : (tables["team_children"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(7695, 5)
Proportion de valeurs manquantes par variables dans la table team_children


,nb,pct
parent_team_id,0,0.0
parent_team_name,0,0.0
child_team_id,0,0.0
child_team_name,0,0.0
_last_modified_at,0,0.0


In [86]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["team_children"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["team_children"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["team_children"].columns[tables["team_children"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  0
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


### `team_competitions_seasons`

In [89]:
print(tables["team_competitions_seasons"].shape)
print("Proportion de valeurs manquantes par variables dans la table team_competitions_seasons")
pd.DataFrame({
    "nb" : tables["team_competitions_seasons"].isna().sum(),
    "pct" : (tables["team_competitions_seasons"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(196378, 7)
Proportion de valeurs manquantes par variables dans la table team_competitions_seasons


,nb,pct
club_id,0,0.00
team_name,0,0.00
season_id,0,0.00
_last_modified_at,0,0.00
competition_name,32189,16.39
competition_id,32189,16.39
club_division,32189,16.39


In [90]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["team_competitions_seasons"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["team_competitions_seasons"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["team_competitions_seasons"].columns[tables["team_competitions_seasons"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  32189
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


### `PLAYER_TEAMMATES_PLAYED_WITH`

In [93]:
print(tables["player_teammates_played_with"].shape)
print("Proportion de valeurs manquantes par variables dans la table player_teammates_played_with")
pd.DataFrame({
    "nb" : tables["player_teammates_played_with"].isna().sum(),
    "pct" : (tables["player_teammates_played_with"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(1257342, 6)
Proportion de valeurs manquantes par variables dans la table player_teammates_played_with


,nb,pct
player_id,0,0.00
teammate_player_id,0,0.00
teammate_player_name,0,0.00
ppg_played_with,37080,2.95
minutes_played_with,804153,63.96
joint_goal_participation,1035177,82.33


In [94]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_teammates_played_with"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_teammates_played_with"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_teammates_played_with"].columns[tables["player_teammates_played_with"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  1232285
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


### `PLAYER INJURY HISTORIES`

In [97]:
print(tables["player_injuries"].shape)
print("Proportion de valeurs manquantes par variables dans la table player_injuries")
pd.DataFrame({
    "nb" : tables["player_injuries"].isna().sum(),
    "pct" : (tables["player_injuries"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(143195, 7)
Proportion de valeurs manquantes par variables dans la table player_injuries


,nb,pct
player_id,0,0.00
season_name,0,0.00
injury_reason,0,0.00
games_missed,0,0.00
from_date,22,0.02
days_missed,22,0.02
end_date,1523,1.06


In [98]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_injuries"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_injuries"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_injuries"].columns[tables["player_injuries"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  1545
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index([], dtype='str')


### `PLAYER_NATIONAL_TEAM_PERFORMANCES`

In [101]:
print(tables["player_national_performances"].shape)
print("Proportion de valeurs manquantes par variables dans la table player_national_performances")
pd.DataFrame({
    "nb" : tables["player_national_performances"].isna().sum(),
    "pct" : (tables["player_national_performances"].isna().mean() * 100 ).round(2)
}).sort_values("pct")

(92701, 9)
Proportion de valeurs manquantes par variables dans la table player_national_performances


,nb,pct
player_id,0,0.00
team_id,0,0.00
matches,0,0.00
goals,0,0.00
shirt_number,0,0.00
career_state,0,0.00
debut_game_id,30347,32.74
coach_id,36922,39.83
debut,92701,100.00


In [102]:
print("Nombre de lignes avec au moins une valeurs manquante : ", tables["player_national_performances"].isna().any(axis=1).sum())
print("Nombre de lignes entièrement vide : ", tables["player_national_performances"].isna().all(axis=1).sum())
print("Colonnes entièrement vide : ", tables["player_national_performances"].columns[tables["player_national_performances"].isna().all()])

Nombre de lignes avec au moins une valeurs manquante :  92701
Nombre de lignes entièrement vide :  0
Colonnes entièrement vide :  Index(['debut'], dtype='str')
